<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_04_feature_engineering/stage_04a_technical_indicators.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_04a_technical_indicators**

## Introducción

Esta notebook genera y evalúa un conjunto de indicadores técnicos intradía para el índice MNQ. A partir del dataset minuto a minuto, se crean objetivos de retorno a distintos horizontes, se calculan indicadores por día evitando mezclar jornadas, se mide su relación con el target mediante Information Coefficient (IC) y se guardan dos datasets listos para análisis y modelado.

Se desarrolla en los siguientes puntos:

0. Configuración del Entorno

    Se prepara el entorno conectando Google Drive y clonando el repositorio de trabajo. Se instalan y cargan las librerías necesarias como pandas, numpy, matplotlib, scipy y ta. Luego se carga el dataset base en formato Parquet llamado mnq_intraday_data, que contiene los datos minuto a minuto del índice MNQ. Se verifican aspectos como número de jornadas, registros por día y consistencia temporal.
    

1. Relación entre indicadores técnicos y el target de retorno
    
    Se define las variables objetivo como los retornos futuros a 30, 60 y 90 minutos, llamados target_return_**. Esta elección se debe a que en análisis previos mostró mejor correlación con los indicadores técnicos. Se utiliza esta métrica como referencia para evaluar si los indicadores calculados aportan información predictiva.

2. Indicadores Técnicos

    Los indicadores se calculan de manera independiente para cada jornada, evitando mezclar datos de diferentes días. Entre los indicadores aplicados se encuentran:

    - RSI en ventanas de 3, 5, 7 y 14 periodos
    - Momentum en ventanas de 3, 5 y 10 periodos
    - Relación de volumen respecto a su media móvil en 15, 20 y 30 periodos
    - MACD diferencial entre la línea MACD y su señal
    - Entre otros como Bandas de Bollingers, ROC, etc.

    Estos indicadores se añaden al dataset como nuevas columnas, generando un conjunto de variables explicativas listo para ser evaluado.

3. Calculo de Information Coefficient (IC)

    Se mide la relación entre cada indicador y el target de retorno a 30, 60 y 90 minutos utilizando el Information Coefficient basado en correlación de Spearman. Para cada indicador se calculan el promedio y la desviación estándar del IC por día, lo que permite identificar la fuerza y estabilidad de la señal. El análisis ayuda a determinar qué indicadores son más relevantes y consistentes como factores predictivos.

4. Guardado de dataset de indicadores técnicos

    El dataset enriquecido, que incluye las columnas OHLCV, los target_return_** y los indicadores técnicos calculados, se guarda en formato Parquet dentro de la carpeta 2_feature_engineering, junto a un archivo IC_technical_indicators. Estos archivos quedan disponibles para posteriores etapas de exploración, selección de variables y entrenamiento de modelos.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [188]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [189]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [190]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

Librería instalada: technical-analysis


### 0.3. Importación de librerías


In [191]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

### 0.4. Definición de rutas



In [192]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [193]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/target_definition_summary.json"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/technical_indicators_summary.json"))

In [194]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.4. Códigos auxiliares para carga de datos y visualización


In [195]:
def load_data():

    # Definir la URL del archivo Parquet en Drive
    data_path = f'{drive_path}/data/processed/mnq_intraday_labeled.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [196]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [197]:
# Carga del JSON
with IN_ARTIFACT.open("r") as f:
    target_definition_summary = json.load(f)

In [198]:
def print_target_definition_summary(
    target_definition_summary: list,
    sort_key: str = "horizon_min",
    line_width: int = 60
) -> None:
    """
    Imprime un resumen legible del target_definition_summary,
    organizado por horizonte.

    Parámetros
    ----------
    target_definition_summary : list[dict]
        Salida del stage_03 con definición de targets.
    sort_key : str
        Clave para ordenar los horizontes (default: horizon_min).
    line_width : int
        Ancho del separador visual.
    """
    for item in sorted(target_definition_summary, key=lambda x: x[sort_key]):
        h = item["horizon_min"]

        print("\n" + "=" * line_width)
        print(f"HORIZONTE: {h} min")
        print("=" * line_width)

        print(f"Δ base (pts):      {item['delta_base_pts']:.2f}")
        print(f"Δ operativo (pts): {item['delta_op_pts']:.2f}")
        print(f"Δ cola (pts):      {item['delta_tail_pts']:.2f}")
        print(f"Targets:           {item['target_columns']}")
        print(f"Filas totales:     {item['n_rows']:,}")
        print(f"Días:              {item['n_days']:,}")
        print(f"Filas/día (μ):     {item['rows_per_day_mean']:.1f}")


In [199]:
#Carga de dataset base:
mnq_intraday_labeled = load_data()
info_dataset(mnq_intraday_labeled)

# Eliminación de filas con NaN
mnq_intraday_labeled = mnq_intraday_labeled.dropna()

# Verificación posterior
info_dataset(mnq_intraday_labeled)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York
Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio 06:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York


## **1. Relación entre indicadores técnicos, information coefficient y los targets definidos del stage_03**

### **1.1. Indicadores técnicos**

Los indicadores técnicos se construyen a partir de la serie de precios intradía, y en particular sobre la variable de cierre (close). Estas transformaciones matemáticas buscan capturar propiedades dinámicas del mercado tales como tendencia, momentum, reversión, volatilidad y estructura temporal del movimiento de precios.

En el marco actual del proyecto, la variable objetivo (target) ya no se define como un retorno normalizado, sino como un movimiento futuro absoluto en puntos (delta en puntos), medido sobre horizontes temporales discretos (60 y 90 minutos). A partir de estos deltas se definen distintos umbrales económicamente relevantes (base, operativo y cola), que dan lugar a señales de trade y targets binarios asociados. Los valores de delta definidos en el stage_03_target_definitin son los siguientes:

In [200]:
print_target_definition_summary(target_definition_summary)


HORIZONTE: 60 min
Δ base (pts):      52.12
Δ operativo (pts): 84.18
Δ cola (pts):      140.70
Targets:           trade_60, target_op_60, target_tail_60
Filas totales:     626,743
Días:              1,303
Filas/día (μ):     481.0

HORIZONTE: 90 min
Δ base (pts):      60.75
Δ operativo (pts): 97.22
Δ cola (pts):      162.30
Targets:           trade_90, target_op_90, target_tail_90
Filas totales:     626,743
Días:              1,303
Filas/día (μ):     481.0


Esto implica que, aunque los indicadores técnicos se calculen directamente sobre el precio, su evaluación no se orienta a explicar la evolución instantánea del close, sino a medir su capacidad para anticipar movimientos futuros de magnitud suficiente, expresados en puntos, dentro de un horizonte temporal determinado.

En consecuencia, el vínculo entre indicadores y targets se establece en términos de poder predictivo sobre la ocurrencia de deltas futuros significativos, es decir, sobre la probabilidad de que el mercado alcance determinados umbrales de movimiento (Δ base, Δ operativo o Δ cola) en el horizonte considerado. El análisis posterior se centra, por tanto, en identificar qué indicadores y configuraciones temporales contienen información relevante para discriminar contextos de no-trade, trade operativo o eventos de cola, coherentes con los targets definidos en el stage_03.

### **1.2. Information Coefficient (IC) versus targets**

En el dataset `mnq_intraday_labeled`, cada fila representa una decisión potencial en un instante $𝑡$ del intradía. Para ese instante, los targets (trade_60, target_op_60, target_tail_60, y sus equivalentes a 90 minutos) indican si, partiendo desde $𝑡$, el precio alcanza o no determinados umbrales de movimiento en un horizonte futuro fijo.

La ventana de gestión (08:20–08:40) no redefine el target ni introduce una nueva variable temporal, sino que delimita el conjunto de instantes $𝑡$
en los cuales el sistema está habilitado a evaluar señales y tomar decisiones. En consecuencia, el análisis se restringe exclusivamente a las filas cuyo timestamp pertenece a dicha ventana horaria.

Por su parte, la llamada ventana de ejecución (por ejemplo, 09:10–09:40 para un horizonte de 60 minutos) no es una entidad explícita del modelo ni del cálculo del IC. Esta ventana surge de manera implícita, ya que corresponde al intervalo temporal donde se materializa el resultado futuro de las decisiones tomadas en la ventana de gestión. Es decir, para cada $𝑡 ∈ [08:10,08:40]$, el target a 60 minutos refleja el comportamiento del precio en $𝑡+60$, que naturalmente cae dentro de esa franja posterior.

Bajo este esquema, el Information Coefficient (IC) se calcula correlacionando, para cada instante $𝑡$ de la ventana de gestión:

$𝑋_{t}$ : el valor del indicador técnico calculado con información disponible hasta $𝑡$

$𝑌_{t,h}$: el target asociado a ese mismo instante 𝑡 y a un horizonte
ℎ (por ejemplo, `delta_pts_60` o `trade_60`).

De este modo, el IC mide directamente la capacidad del indicador, evaluado en el momento de decisión, para anticipar la ocurrencia de un movimiento futuro económicamente relevante. No se comparan ventanas horarias entre sí, ni se agregan bloques temporales: la relación se establece fila a fila, respetando estrictamente la causalidad temporal.

En términos operativos, un IC positivo indica que ciertos estados del mercado, caracterizados por los indicadores técnicos en la ventana de gestión, están sistemáticamente asociados a una mayor probabilidad de alcanzar los targets definidos en el stage_03. Esto justifica su uso como variables explicativas en el entrenamiento del modelo predictivo.

### **1.3. Alineación entre el punto 8 (stage_03b) y el cálculo del IC (stage_04)**

La construcción de las ventanas operativas desarrollada en el punto 8 del stage_03b establece una separación conceptual fundamental entre:

- una ventana de gestación (predicción), donde se origina la información anticipatoria, y

- una ventana de expansión (ejecución), donde los movimientos alcanzan magnitud económica explotable.

Esta separación no entra en conflicto con la definición de los targets ni con el cálculo del Information Coefficient (IC); por el contrario, ambos enfoques son complementarios y coherentes, siempre que se entienda correctamente el rol temporal de cada elemento.



#### **1.3.1. Nivel estadístico (dataset y targets)**


En `mnq_intraday_labeled`, los targets (`delta_pts_h`, `trade_h`, `target_op_h`, `target_tail_h`) están definidos fila a fila, para cada instante t, como el resultado del movimiento futuro observado en t+h.

Esto significa que:
- el target está anclado temporalmente al instante t
- el horizonte h determina cuándo se materializa el resultado,
- no existen targets definidos “por ventana”, sino por decisión potencial en un minuto específico.

Cuando el análisis se restringe a la ventana de gestación (por ejemplo, 08:20-08:40), lo que se hace es seleccionar el subconjunto de instantes
t que, según el análisis empírico del punto 8, concentran información anticipatoria relevante.

En este contexto, el IC mide:
 - La relación estadística entre el estado del mercado en t (capturado por los indicadores técnicos) y el resultado futuro asociado a ese mismo t, que se materializa en la ventana de expansión.

#### **1.3.2. Nivel operativo (modelo y ejecución)**

Desde el punto de vista operativo, el esquema temporal se interpreta de la siguiente manera:

- Ventana de gestación (08:20-08:40)
  - Se calculan los indicadores técnicos.
  - El modelo evalúa si, desde esos instantes, es probable alcanzar un delta relevante a 60 o 90 minutos.
  - Aquí reside la capacidad predictiva.

- Ventana de expansión (09:10-09:40)
  - Es el período donde, empíricamente, los movimientos alcanzan mayor frecuencia y magnitud.
  - Las predicciones generadas previamente habilitan (o no) la toma de operaciones reales.
  - El punto de entrada puede ubicarse en cualquier minuto de esta franja, sujeto a reglas operativas adicionales.

- Horizonte de resultado
- El cierre de la operación ocurre según:
- `delta_op` (objetivo operativo),
- `delta_tail` (extensión),
- o reglas de stop,
  
  siempre respetando el horizonte temporal definido desde el instante de entrada.

#### **1.3.3. Rol del IC dentro de este esquema**

El **Information Coefficient** no evalúa la ejecución, sino la calidad de la información generada en la ventana de gestación.

Su función es responder a la pregunta:

  ¿Los indicadores técnicos calculados en la ventana de gestación contienen información útil para anticipar los movimientos que se expanden y se monetizan más adelante?

Por lo tanto:
- El IC se calcula exclusivamente en la ventana de gestación.
- Los targets ya incorporan, de forma implícita, el desfase temporal hacia la ventana de expansión.
- La coherencia temporal y la ausencia de data leakage quedan garantizadas por construcción.

#### **Conclusión sintética**

La lógica del point 8 define dónde nace la información y dónde se ejecuta la operación.

El cálculo del IC, en el stage_04, cuantifica qué tan informativa es esa ventana de gestación respecto de los resultados que se materializan posteriormente.

Ambos enfoques describen el mismo fenómeno desde niveles distintos (operativo vs estadístico) y están plenamente alineados dentro del diseño del pipeline.

## **2. Indicadores Técnicos**

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

### 2.1. **RSI (Relative Strength Index)**

Mide la fuerza relativa del precio en los últimos períodos (3, 5, 7, 14), oscilando entre 0 y 100.

  - Valores altos indican posibles condiciones de sobrecompra, mientras que valores bajos sugieren sobreventa.
  
  - Calculado sobre los precios de cierre intradía, el RSI es útil para identificar puntos de reversión potenciales en el corto plazo.

In [201]:
def calcular_rsi(df=mnq_intraday_labeled, target='close' ):
  rsi_columns = ['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_5'] = ta.momentum.RSIIndicator(grupo[target], window=5).rsi()
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, rsi_columns

### 2.2. **Momentum**

Mide la aceleración reciente del precio mediante la variación porcentual entre el precio actual y el de hace N minutos.

  - Un valor positivo indica una subida reciente, lo que podría sugerir una continuación alcista.

  - Un valor negativo señala presión bajista reciente, potencialmente anticipando una continuación a la baja.

In [202]:
def calcular_momentum(df=mnq_intraday_labeled, target='close' ):
  momentum_columns = ['momentum_10', 'momentum_5','momentum_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['momentum_10'] = grupo[target].pct_change(10)
        grupo['momentum_5'] = grupo[target].pct_change(5)
        grupo['momentum_3'] = grupo[target].pct_change(3)
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, momentum_columns

### 2.3. **Relación de volumen actual vs. su promedio reciente**

Compara el volumen actual con su media móvil en distintas ventanas de tiempo: 15, 20 y 30 minutos.

  - Un valor mayor a 1 indica un volumen superior al promedio de la ventana correspondiente, lo que puede reflejar interés creciente o actividad institucional.

  - Un valor menor a 1 sugiere baja actividad o consolidación del precio.

Esta métrica permite detectar aumentos de volumen ("spikes") sin depender del volumen en crudo, y las diferentes ventanas permiten capturar variaciones en la dinámica de corto plazo con distinta sensibilidad.


In [203]:
def calcular_volumen_ratio(df=mnq_intraday_labeled, target='close'):
  volume_ratio_columns = ['volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['volume_ratio_15'] = grupo['volume'] / grupo['volume'].rolling(15).mean()
        grupo['volume_ratio_20'] = grupo['volume'] / grupo['volume'].rolling(20).mean()
        grupo['volume_ratio_30'] = grupo['volume'] / grupo['volume'].rolling(30).mean()
        grupo['volume_ratio_60'] = grupo['volume'] / grupo['volume'].rolling(60).mean()
        grupo['volume_ratio_90'] = grupo['volume'] / grupo['volume'].rolling(90).mean()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

  return df, volume_ratio_columns

### 2.4. **MACD diferencial (señal de cruce)**

Representa la diferencia entre la línea MACD y su línea de señal (una media exponencial de sí misma).

  - Un valor positivo y creciente indica momentum alcista.

  - Un valor negativo sugiere presión bajista.
  
Es ampliamente utilizado para detectar giros de tendencia y cambios en la dinámica del mercado.


In [204]:
def calcular_macd(df=mnq_intraday_labeled, target='close'):
    macd_columns = ['macd']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['macd'] = ta.trend.MACD(grupo[target]).macd_diff()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, macd_columns

### 2.5. **Distancia del precio actual a su EMA (15, 20 y 30 minutos)**

Mide el desvío porcentual del precio respecto a su media exponencial en diferentes ventanas, y actúa como indicador de sobreextensión o retorno a la media.

  - Si el precio está muy por encima de la EMA, puede anticipar una reversión bajista o una posible aceleración alcista.

  - Si está por debajo, podría indicar agotamiento o presión vendedora.<br>

Esta métrica se expresa como un porcentaje relativo, lo que facilita la comparación entre distintas ventanas temporales y condiciones de mercado.

Usar varias ventanas (15, 20 y 30 minutos) permite capturar diferentes horizontes de reacción del precio frente a su media móvil.


In [205]:
def calcular_ema(df=mnq_intraday_labeled, target='close'):
    ema_columns = ['price_ema15', 'price_ema20', 'price_ema30',  'price_ema60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['price_ema15'] = grupo[target] / grupo[target].ewm(span=15).mean() - 1
        grupo['price_ema20'] = grupo[target] / grupo[target].ewm(span=20).mean() - 1
        grupo['price_ema30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1
        grupo['price_ema60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, ema_columns

### 2.6. **%K Estocástico**

Mide la posición relativa del precio actual dentro del rango alto-bajo de los últimos n periodos (generalmente 14).

  - Se utiliza para identificar condiciones extremas de sobrecompra o sobreventa.

  - Un valor cercano a 100 indica que el precio está cerca del máximo reciente (potencial sobrecompra), mientras que un valor cercano a 0 indica proximidad al mínimo reciente (posible sobreventa).

Es útil para detectar momentos en los que el precio puede estar excesivamente extendido y susceptible a una reversión.


In [206]:
def calcular_stochastic(df=mnq_intraday_labeled, target='close'):
    stoch_columns = ['stoch_k_14', 'stoch_k_20', 'stoch_k_30']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        stoch_14 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=14, smooth_window=3
        )
        grupo['stoch_k_14'] = stoch_14.stoch()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo['stoch_k_20'] = stoch_20.stoch()

        stoch_30 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=30, smooth_window=3
        )
        grupo['stoch_k_30'] = stoch_30.stoch()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, stoch_columns


### 2.7.**%B de Bollinger (Bollinger Band Percent)**

Indica la posición del precio actual en relación con las bandas de Bollinger, que están construidas alrededor de una media móvil usando desviaciones estándar.

  - Un valor de %B > 1 sugiere que el precio está por encima de la banda superior, lo que podría implicar exceso de optimismo o momentum fuerte.

  - Un valor < 0 indica que está por debajo de la banda inferior, posible señal de pánico o sobreventa extrema.

Este indicador es eficaz para identificar zonas de congestión, breakout o reversiones basadas en la volatilidad reciente.


In [207]:
from ta.volatility import BollingerBands

def calcular_bollinger(df=mnq_intraday_labeled, target='close'):
    '''bollinger_columns = [
        'bb_percent_15_15', 'bb_percent_20_15', 'bb_percent_30_15',
        'bb_percent_15_20', 'bb_percent_20_20', 'bb_percent_30_20',
        'bb_percent_15_25', 'bb_percent_20_25', 'bb_percent_30_25',
    ]'''

    bollinger_columns = [
        'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15',
        'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20',
        'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 1.5
        grupo['bb_15_15'] = BollingerBands(grupo[target], window=15, window_dev=1.5).bollinger_pband()
        grupo['bb_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_30_15'] = BollingerBands(grupo[target], window=30, window_dev=1.5).bollinger_pband()
        grupo['bb_60_15'] = BollingerBands(grupo[target], window=60, window_dev=1.5).bollinger_pband()

        # std: 2
        grupo['bb_15_20'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60_20'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()

        # std: 2.5
        grupo['bb_15_25'] = BollingerBands(grupo[target], window=15, window_dev=2.5).bollinger_pband()
        grupo['bb_20_25'] = BollingerBands(grupo[target], window=20, window_dev=2.5).bollinger_pband()
        grupo['bb_30_25'] = BollingerBands(grupo[target], window=30, window_dev=2.5).bollinger_pband()
        grupo['bb_60_25'] = BollingerBands(grupo[target], window=60, window_dev=2.5).bollinger_pband()



        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns


In [208]:
def calcular_bollinger_resume(df=mnq_intraday_labeled, target='close'):

    bollinger_columns = [
        'bb_15', 'bb_20', 'bb_30', 'bb_60',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 2
        grupo['bb_15'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns

### 2.8. **ATR normalizado (Average True Range / precio)**

Representa la volatilidad absoluta reciente ajustada al nivel del precio.

  - El ATR mide el rango promedio de oscilación de un activo en los últimos n periodos, capturando tanto movimientos bruscos como gaps.

  - Al normalizarlo dividiéndolo por el precio, se obtiene una medida relativa, comparable entre distintos niveles de mercado.

Este indicador es útil para detectar momentos de alta o baja volatilidad intradía, que pueden influir en la confiabilidad de otras señales técnicas.


In [209]:
def calcular_atr(df=mnq_intraday_labeled, target='close', windows=[5, 10, 14, 20, 30]):
    atr_columns = []

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        for w in windows:
            col_name = f'atr_norm_{w}'
            atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=w)
            grupo[col_name] = atr.average_true_range() / grupo[target]
            atr_columns.append(col_name)
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, list(set(atr_columns))

In [210]:
def calcular_atr_14(df=mnq_intraday_labeled, target='close'):
    atr_columns = ['atr_norm']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=14)
        grupo['atr'] = atr.average_true_range()
        grupo['atr_norm'] = grupo['atr'] / grupo[target]
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, atr_columns

### 2.9. **ROC (Rate of Change)**

Calcula la tasa de cambio porcentual del precio con respecto a su valor n minutos atrás.

- Es un indicador de momentum que capta aceleraciones o desaceleraciones recientes del precio.

- Valores positivos indican presión alcista; negativos, presión bajista.

A diferencia del momentum tradicional, el ROC expresa el cambio de forma normalizada y en porcentaje, lo que facilita su interpretación comparativa entre distintos activos o marcos temporales.


In [211]:
def calcular_roc(df=mnq_intraday_labeled, target='close'):
    roc_columns = ['roc_5', 'roc_10', 'roc_20','roc_30','roc_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_10'] = ROCIndicator(close=grupo[target], window=10).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, roc_columns

### 2.10. **Cálculo final de indicadores técnicos**

Calculamos los indicadores técnicos

In [212]:
mnq_intraday_with_indicators = mnq_intraday_labeled.copy()

mnq_intraday_with_indicators, rsi_columns = calcular_rsi(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, momentum_columns = calcular_momentum(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, volume_ratio_columns = calcular_volumen_ratio(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, macd_columns = calcular_macd(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, ema_columns = calcular_ema(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, stoch_columns = calcular_stochastic(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, bollinger_columns = calcular_bollinger(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, atr_columns = calcular_atr(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, roc_columns = calcular_roc(mnq_intraday_with_indicators)

Construimos el listado de indicadores técnicos

In [213]:
indicator_columns = (
    rsi_columns
    + momentum_columns
    + volume_ratio_columns
    + macd_columns
    + ema_columns
    + stoch_columns
    + bollinger_columns
    + atr_columns
    + roc_columns
)

Filtramos todos los NaNs del dataset

In [214]:
info_dataset(mnq_intraday_with_indicators)

# Eliminación de filas con NaN
mnq_intraday_with_indicators = mnq_intraday_with_indicators.dropna()

# Verificación posterior
info_dataset(mnq_intraday_with_indicators)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio 06:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York
Información del dataset:

	Cantidad de días: 1303
	Registros por día: 392
	Hora diaria de inicio 07:59
	Hora diaria de final 14:30
	Zona horaria: America/New_York


In [215]:
assert not mnq_intraday_with_indicators.isna().any().any(), \
    "El dataset contiene NaN"

##**3. Calculo de Information Coefficient (IC)**

### **3.1. Tipos de targets y criterios para el cálculo del Information Coefficient (IC)**

En el dataset `mnq_intraday_labeled` coexisten distintos tipos de variables objetivo, que responden a naturalezas estadísticas y operativas diferentes. En consecuencia, el cálculo e interpretación del Information Coefficient (IC) debe adaptarse a cada caso, manteniendo un criterio metodológico coherente.

#### 3.1.1. Targets continuos `delta_pts_h`

Las variables delta_pts_h representan el movimiento futuro absoluto en puntos, medido desde un instante 𝑡 hasta 𝑡+ℎ. Este tipo de target constituye el caso más directo y conceptualmente “puro” para el cálculo del IC.

En este contexto, el IC mide si un indicador técnico es capaz de ordenar correctamente la dirección y la magnitud relativa de los movimientos futuros. El coeficiente recomendado es la correlación de Spearman, ya que:
- no asume relaciones lineales,
- es robusta frente a valores extremos,
- evalúa asociaciones monotónicas, más adecuadas para series financieras.

La interpretación es directa:

- un IC positivo indica que valores más altos del indicador tienden a asociarse con deltas futuros mayores,
- un IC negativo indica la relación inversa.

Por estas razones, `delta_pts_h` se adopta como target principal para el análisis de IC.

#### 3.1.2. Targets discretos ordinales: `trade_h` (-1, 0, +1)

La variable `trade_h` codifica el sentido operativo del movimiento futuro, distinguiendo entre posiciones cortas (-1), ausencia de trade (0) y posiciones largas (+1). Se trata de un target ordinal, no continuo.

En este caso, el IC evalúa si el indicador tiende a tomar valores sistemáticamente mayores o menores a medida que el sentido del trade progresa desde short hacia long. Nuevamente, la correlación de Spearman resulta apropiada, ya que respeta el orden implícito de las categorías.

La interpretación es la siguiente:

- un IC positivo indica que el indicador suele ser mayor en escenarios long que en no-trade y short,
- un IC negativo indica el patrón opuesto.

Dado que el estado “no-trade” suele ser dominante en frecuencia, este target puede generar ICs atenuados. Por este motivo, el análisis puede complementarse con evaluaciones restringidas al subconjunto de observaciones donde `trade_h ≠ 0`, con el fin de aislar el componente puramente direccional.

#### 3.1.3. Targets binarios: `target_op_h` y `target_tail_h` (0/1)


Las variables `target_op_h` y `target_tail_h` indican la ocurrencia o no de eventos económicamente relevantes (alcance del objetivo operativo o de cola). Se trata de targets binarios, asociados a eventos discretos.

En este contexto, el IC mide si el indicador técnico tiende a tomar valores más altos (o más bajos) cuando el evento ocurre (1) frente a cuando no ocurre (0). Aunque el target es binario, la correlación de Spearman sigue siendo válida como medida de asociación monotónica.

La interpretación es:

- IC positivo: valores altos del indicador se asocian a mayor probabilidad de alcanzar el objetivo,
- IC negativo: valores altos del indicador se asocian a menor probabilidad del evento.

Dado que estos eventos pueden ser poco frecuentes —especialmente en el caso de `target_tail_h`—, el IC puede presentar mayor variabilidad. Por ello, se recomienda calcular el IC por día y luego resumirlo mediante estadísticas agregadas (media, mediana y dispersión).

#### 3.1.4. Recomendación operativa


Para cada indicador técnico y cada horizonte ℎ, el análisis de IC se estructura en tres niveles complementarios:

- IC principal: asociación entre el indicador y `delta_pts_h`.
- IC direccional: asociación entre el indicador y `trade_h`.
- IC de evento: asociación entre el indicador y `target_op_h` y `target_tail_h`.

Este enfoque permite evaluar, de manera consistente, la relación entre los indicadores técnicos y la magnitud, dirección y relevancia económica de los movimientos futuros, respetando la estructura temporal y operativa definida en las etapas previas del pipeline.

### **3.2. Implementación de cálculo de IC**

#### **3.2.1. Código**

In [216]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

#=========================
#Utilidades base
#=========================

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    """
    Calcula el Information Coefficient (IC) usando correlación de Spearman.

    - x: indicador técnico en el instante t
    - y: target asociado a ese mismo t (delta, trade, evento)
    - Ignora valores NaN en ambas series
    - Devuelve np.nan si no hay suficientes observaciones
    """

    # Máscara para quedarnos solo con pares (x, y) válidos
    mask = x.notna() & y.notna()

    # Si hay menos de 3 puntos válidos, la correlación no es confiable
    if mask.sum() < 3:
        return np.nan

    # Correlación de Spearman (rank correlation)
    return spearmanr(x[mask], y[mask]).correlation

def filter_time_window(
    df: pd.DataFrame,
    start="08:10",
    end="08:50"
) -> pd.DataFrame:
    """
    Filtra el DataFrame por una ventana horaria intradía.

    - Se utiliza para aislar la ventana de gestación (predicción)
    - Requiere que el índice del DataFrame sea DatetimeIndex
    """
    return df.between_time(start, end)

def daily_ic(
    df: pd.DataFrame,
    indicator_col: str,
    target_col: str
) -> pd.Series:
    """
    Calcula el IC de Spearman por día.

    - Agrupa el dataset por la columna 'date'
    - Dentro de cada día, calcula IC(indicador, target)
    - Devuelve una serie con un IC por jornada
    """
    return df.groupby("date").apply(
        lambda g: spearman_ic(g[indicator_col], g[target_col])
    )

#=========================
#Tabla IC (indicadores × horizontes)
#=========================

def compute_ic_table(
    df: pd.DataFrame,
    indicator_columns: list,
    horizons=(60, 90),
    window_start="08:10",
    window_end="08:50",
    use_daily_ic: bool = True
) -> pd.DataFrame:
    """
    Calcula una tabla resumen de Information Coefficient (IC) para cada
    indicador técnico y cada horizonte temporal (60 / 90).

    Para cada indicador y horizonte se evalúa su relación con:
    - delta_pts_h        → magnitud del movimiento futuro (continuo)
    - trade_h            → dirección (-1, 0, +1)
    - trade_h_only       → dirección pura (excluye no-trade)
    - target_op_h        → evento operativo (0 / 1)
    - target_tail_h      → evento extremo (0 / 1)

    Parámetro clave:
    - use_daily_ic = True
        Calcula IC por día y luego promedia (más robusto estadísticamente)
    - use_daily_ic = False
        Calcula IC global usando todas las filas juntas
    """

    # 1. Filtramos el dataset únicamente a la ventana de gestación
    dfw = filter_time_window(df, window_start, window_end).copy()

    # Lista donde se acumularán los resultados fila por fila
    rows = []

    # 2. Iteramos por cada indicador técnico
    for ind in indicator_columns:

        # 3. Iteramos por cada horizonte temporal (60 y 90 minutos)
        for h in horizons:

            # Nombres de las columnas target asociadas a ese horizonte
            cols = {
                "delta": f"delta_pts_{h}",
                "trade": f"trade_{h}",
                "op": f"target_op_{h}",
                "tail": f"target_tail_{h}",
            }

            # 4. Verificamos que existan todas las columnas necesarias
            missing = [
                c for c in [ind, *cols.values()]
                if c not in dfw.columns
            ]

            # Si falta alguna columna, devolvemos NaN y seguimos
            if missing:
                rows.append({
                    "indicator": ind,
                    "horizon": h,
                    "IC_delta": np.nan,
                    "IC_trade": np.nan,
                    "IC_trade_only": np.nan,
                    "IC_target_op": np.nan,
                    "IC_target_tail": np.nan,
                    "note": f"missing: {missing}"
                })
                continue

            # 5. Cálculo del IC (modo recomendado: diario)
            if use_daily_ic:

                # IC principal: indicador vs delta futuro
                ic_delta = daily_ic(dfw, ind, cols["delta"]).mean()

                # IC direccional: indicador vs trade (-1, 0, +1)
                ic_trade = daily_ic(dfw, ind, cols["trade"]).mean()

                # IC direccional puro: solo cuando hay trade
                df_trade_only = dfw[dfw[cols["trade"]] != 0]
                ic_trade_only = (
                    daily_ic(df_trade_only, ind, cols["trade"]).mean()
                    if len(df_trade_only) > 0 else np.nan
                )

                # IC contra evento operativo
                ic_op = daily_ic(dfw, ind, cols["op"]).mean()

                # IC contra evento extremo
                ic_tail = daily_ic(dfw, ind, cols["tail"]).mean()

            # 6. Alternativa: IC global (no recomendado, pero disponible)
            else:
                ic_delta = spearman_ic(dfw[ind], dfw[cols["delta"]])
                ic_trade = spearman_ic(dfw[ind], dfw[cols["trade"]])

                df_trade_only = dfw[dfw[cols["trade"]] != 0]
                ic_trade_only = (
                    spearman_ic(
                        df_trade_only[ind],
                        df_trade_only[cols["trade"]]
                    ) if len(df_trade_only) > 0 else np.nan
                )

                ic_op = spearman_ic(dfw[ind], dfw[cols["op"]])
                ic_tail = spearman_ic(dfw[ind], dfw[cols["tail"]])

            # 7. Guardamos los resultados para este indicador y horizonte
            rows.append({
                "indicator": ind,
                "horizon": h,
                "IC_delta": ic_delta,
                "IC_trade": ic_trade,
                "IC_trade_only": ic_trade_only,
                "IC_target_op": ic_op,
                "IC_target_tail": ic_tail,
                "note": ""
            })

    # 8. Construimos el DataFrame final
    out = pd.DataFrame(rows)

    # 9. Ranking principal: magnitud del IC contra delta
    out["abs_IC_delta"] = out["IC_delta"].abs()

    out = (
        out
        .sort_values(["horizon", "abs_IC_delta"], ascending=[True, False])
        .reset_index(drop=True)
    )

    return out

#### **3.2.2. Aplicación**

In [187]:
from pathlib import Path
import pandas as pd

# Ruta del artifact de IC
#IC_ARTIFACT_PATH = Path("reports/ic_table.parquet")
IC_ARTIFACT_PATH = DRIVE_DIR / "reports" / "ic_table.parquet"

# ------------------------------------------------------------
# Verificación previa: ¿ya existe la tabla de IC?
# ------------------------------------------------------------
if IC_ARTIFACT_PATH.exists():
    print(f"IC table encontrada. Cargando desde: {IC_ARTIFACT_PATH.resolve()}")
    ic_table = pd.read_parquet(IC_ARTIFACT_PATH)

else:
    print("IC table no encontrada. Se procederá a calcularla.")

    # ------------------------------------------------------------------
    # Cálculo de la tabla de Information Coefficient (IC)
    # ------------------------------------------------------------------
    ic_table = compute_ic_table(
        df=mnq_intraday_with_indicators,     # Dataset intradía con indicadores + targets
        indicator_columns=indicator_columns, # Lista de indicadores técnicos
        horizons=(60, 90),                    # Horizontes de predicción
        window_start="08:10",                 # Inicio ventana de gestación
        window_end="08:40",                   # Fin ventana de gestación
        use_daily_ic=True                     # IC diario promedio (recomendado)
    )

    # Guardado del artifact para no recalcular
    IC_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    ic_table.to_parquet(IC_ARTIFACT_PATH)

    print(f"IC table calculada y guardada en: {IC_ARTIFACT_PATH.resolve()}")

# Visualización rápida
#ic_table.head(20)


IC table no encontrada. Se procederá a calcularla.
IC table calculada y guardada en: /content/drive/MyDrive/neural_profit/reports/ic_table.parquet


#### **3.2.3. Resultado**

In [217]:
ic_table

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta
0,price_ema60,60,-0.466803,-0.282841,-0.636263,0.015617,0.067863,,0.466803
1,price_ema30,60,-0.443975,-0.270568,-0.629694,0.013366,0.064570,,0.443975
2,bb_60_15,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
3,bb_60_20,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
4,bb_60_25,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
...,...,...,...,...,...,...,...,...,...
79,volume_ratio_15,90,0.013434,0.024266,0.069968,-0.003612,-0.007500,,0.013434
80,volume_ratio_30,90,0.013173,0.025973,0.101723,-0.024943,-0.031617,,0.013173
81,volume_ratio_20,90,0.011937,0.025551,0.096460,-0.011576,-0.017551,,0.011937
82,volume_ratio_90,90,0.010410,0.020027,0.086906,-0.023809,-0.044796,,0.010410


### **3.3. Análisis de resultados**

In [243]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [244]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

#### **3.3.1. Separación por horizonte**

##### **Horizonte 60min**

In [248]:
ic_60 = ic_table[ic_table["horizon"] == 60].copy()
ic_60.head()

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta
0,price_ema60,60,-0.466803,-0.282841,-0.636263,0.015617,0.067863,,0.466803
1,price_ema30,60,-0.443975,-0.270568,-0.629694,0.013366,0.064570,,0.443975
2,bb_60_15,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
3,bb_60_20,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
4,bb_60_25,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702


**Comentarios de ic_60**

- Señal muy fuerte en magnitud: los mejores indicadores tienen |IC_delta| entre 0.33 y 0.47 (muy alto para intradía).
- Signo dominante negativo (reversión): casi todos los indicadores relevantes tienen IC_delta < 0 y también IC_trade_only < 0 ⇒ extensiones en gestación tienden a agotarse/revertir en los próximos 60 min.
- Coherencia direccional fuerte: en los top indicadores, IC_trade_only es incluso más extremo (ej. ~-0.64) que IC_delta ⇒ cuando hay trade, la dirección está muy bien alineada con la señal.
- Redundancia evidente: muchas columnas repiten exactamente el mismo IC (BB_60_15/20/25; BB_30_15/20/25; etc.) ⇒ son variantes altamente correlacionadas; luego conviene quedarse con una por familia.
- Volatilidad/volumen aportan poco a delta: atr_norm_* y volume_ratio_* tienen abs_IC_delta muy bajo (~0.01–0.036) ⇒ en gestación, explican poco la magnitud a 60 min (aunque pueden servir como “contexto” más adelante).

In [245]:
ic_60_top_by_family = select_top_by_family(
    ic_60,
    top_n=1  # 1 indicador por familia
)

ic_60_top_by_family[
    ["family", "indicator", "IC_delta", "IC_trade_only", "abs_IC_delta"]
]

,family,indicator,IC_delta,IC_trade_only,abs_IC_delta
0,trend_price,price_ema60,-0.466803,-0.636263,0.466803
1,volatility_extension,bb_60_15,-0.425702,-0.569178,0.425702
2,momentum_oscillator,rsi_14,-0.422538,-0.612055,0.422538
3,momentum,roc_60,-0.392433,-0.581905,0.392433
4,trend_momentum,macd,-0.333890,-0.535455,0.333890
5,volatility,atr_norm_5,0.035872,-0.067085,0.035872
6,volume,volume_ratio_60,0.012200,-0.000909,0.012200


- Las 5 primeras familias concentran casi toda la señal útil.
  
  `trend_price`, `volatility_extension`, `momentum_oscillator`, `momentum` y `trend_momentum` muestran `abs_IC_delta` muy altos (≈ 0.33–0.47).

- Régimen claramente de reversión.
  
  En esas familias, `IC_delta < 0` y `IC_trade_only < 0` ⇒ extensiones tempranas anticipan agotamiento/reversión en los próximos 60 min.

- Coherencia magnitud-dirección fuerte.

  `IC_trade_only` es incluso más extremo que `IC_delta` (ej. EMA, RSI, ROC), lo que indica alineación direccional cuando hay trade.

- Redundancia bien resuelta.

  Quedarse con 1 indicador por familia elimina duplicaciones (BB/EMA/ROC) sin perder señal.

- Volatilidad y volumen aportan poco a la magnitud.

  `atr_norm_*` y `volume_ratio_*` tienen `abs_IC_delta` muy bajo (≈ 0.01–0.04). No son buenos predictores primarios de delta a 60 min; pueden quedar como contexto, no como features principales.

- Resultado práctico (60 min):

  Un set compacto y fuerte de 5 indicadores (EMA, BB, RSI, ROC, MACD) captura la señal anticipatoria dominante; volatilidad/volumen no son prioritarios para explicar la magnitud del movimiento.

##### **Horizonte 90min**

In [250]:
ic_90 = ic_table[ic_table["horizon"] == 90].copy()
ic_90.head()

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta
42,price_ema60,90,-0.304082,-0.247576,-0.583596,-0.005852,0.033802,,0.304082
43,price_ema30,90,-0.291550,-0.236135,-0.587207,-0.007606,0.023692,,0.291550
44,price_ema20,90,-0.281995,-0.226898,-0.584788,-0.010037,0.016000,,0.281995
45,rsi_14,90,-0.277105,-0.225905,-0.572693,-0.004575,0.022853,,0.277105
46,bb_60_15,90,-0.276758,-0.227443,-0.569022,-0.002400,0.033093,,0.276758


**Comentarios de ic_90**

- Señal fuerte pero atenuada en magnitud: los principales indicadores presentan |IC_delta| entre 0.20 y 0.30, inferiores a 60 min pero aún elevados para intradía.

- Signo dominante negativo (reversión): al igual que en 60 min, casi todos los indicadores relevantes muestran `IC_delta` < 0 y `IC_trade_only` < 0, confirmando que las extensiones tempranas del precio anticipan agotamiento/reversión en el horizonte de 90 min.

- Coherencia direccional mantenida: aunque `IC_trade_only` es algo menor que en 60 min, sigue siendo consistente y del mismo signo que `IC_delta`, lo que indica que la dirección del trade continúa alineada con la señal anticipatoria.

- Persistencia del conjunto informativo: los indicadores más relevantes son los mismos que en 60 min (EMA, BB, RSI, ROC, MACD), lo que sugiere estabilidad del régimen y ausencia de un nuevo mecanismo dominante al extender el horizonte.

- Redundancia nuevamente evidente: se repite la presencia de múltiples variantes con IC idénticos (familias BB, EMA, ROC), confirmando la conveniencia de filtrar por familia para evitar duplicaciones.

- Volatilidad y volumen siguen sin explicar la magnitud: `atr_norm_*` y `volume_ratio_*` mantienen valores de abs_IC_delta muy bajos (≈ 0.01–0.03), indicando que, aun a 90 min, no son buenos predictores primarios de delta, aunque pueden aportar información contextual secundaria.

In [251]:
ic_90_top_by_family = select_top_by_family(
    ic_90,
    top_n=1  # 1 indicador por familia
)

ic_90_top_by_family[
    ["family", "indicator", "IC_delta", "IC_trade_only", "abs_IC_delta"]
]

,family,indicator,IC_delta,IC_trade_only,abs_IC_delta
0,trend_price,price_ema60,-0.304082,-0.583596,0.304082
1,momentum_oscillator,rsi_14,-0.277105,-0.572693,0.277105
2,volatility_extension,bb_60_15,-0.276758,-0.569022,0.276758
3,momentum,roc_30,-0.255689,-0.513596,0.255689
4,trend_momentum,macd,-0.226918,-0.465284,0.226918
5,volatility,atr_norm_5,0.030633,0.001658,0.030633
6,volume,volume_ratio_15,0.013434,0.069968,0.013434


- Las 5 primeras familias concentran la señal relevante.

  `trend_price`, `momentum_oscillator`, `volatility_extension`, `momentum` y `trend_momentum` presentan `abs_IC_delta` moderados-altos (≈ 0.23-0.30), suficientes para un horizonte intradía extendido.

- Régimen de reversión sostenido.

  En estas familias, tanto `IC_delta` como `IC_trade_only` son negativos, lo que indica que las extensiones tempranas del precio en la ventana de gestación anticipan agotamiento o reversión en el horizonte de 90 minutos.

- Coherencia magnitud-dirección preservada.

  Aunque `IC_delta` es menor que en 60 min (atenuación esperable), `IC_trade_only` mantiene magnitudes elevadas (≈ -0.46 a -0.58), confirmando que cuando hay trade, la dirección sigue bien alineada con la señal.

- Redundancia correctamente eliminada por familia.

  La selección de un único indicador por familia (EMA, BB, RSI, ROC, MACD) conserva la información esencial y evita duplicaciones estructurales observadas en el ranking completo.

  - Volatilidad y volumen siguen sin explicar la magnitud.

  `atr_norm_*` y `volume_ratio_*` muestran `abs_IC_delta` muy bajos (≈ 0.01-0.03), lo que indica que no aportan capacidad explicativa primaria sobre el delta a 90 min; su rol potencial es contextual, no predictivo principal.

- Resultado práctico (90 min):

  Un set compacto de 5 indicadores (EMA, BB, RSI, ROC, MACD) captura la señal anticipatoria dominante también en 90 minutos, con menor intensidad que en 60 min pero con consistencia direccional y económica.

##### **Comparación de resultados por horizonte (60 vs 90)**

- Mismo conjunto de familias relevantes.

  En ambos horizontes, las familias dominantes son exactamente las mismas:
  `trend_price`, `volatility_extension`, `momentum_oscillator`, `momentum` y `trend_momentum`
  .
  ⇒ No aparece un régimen nuevo al extender el horizonte.

- Señal más intensa en 60 min.

  - 60 min: `abs_IC_delta` ≈ 0.33-0.47
  - 90 min: `abs_IC_delta` ≈ 0.23-0.30

  ⇒ La señal es más fuerte y concentrada a 60 min; a 90 min se atenúa, como es esperable.

- Régimen económico idéntico: reversión.

  En ambos horizontes:

  - `IC_delta` < 0
  - `IC_trade_only` < 0

  ⇒ Las extensiones tempranas del precio anticipan agotamiento/reversión, no continuación.

- Coherencia direccional sostenida.

  Aunque el `IC_delta` cae en 90 min, `IC_trade_only` sigue siendo alto y del mismo signo.

  ⇒ Cuando hay trade, la dirección continúa bien alineada con la señal en ambos horizontes.

- Redundancia y filtrado por familia válidos en ambos casos.

  El filtrado por familia resuelve la duplicación estructural (EMA, BB, ROC) tanto en 60 como en 90, sin pérdida de información.

- Volatilidad y volumen consistentemente secundarios.

  En ambos horizontes, `atr_norm_*` y `volume_ratio_*` presentan abs_IC_delta muy bajo.

  ⇒ No explican la magnitud del movimiento; su rol potencial es contextual, no principal.

**Conclusión comparativa**

- 60 min ofrece mayor potencia predictiva.
- 90 min mantiene la misma estructura de señal, pero con menor intensidad y mayor ruido.
- Ambos horizontes describen el mismo fenómeno, con distinta extensión temporal.

#### **3.3.2. Primer filtro: fuerza miníma de señal**

Para el análisis intradía se adopta el siguiente criterio empírico de interpretación del Information Coefficient (IC):

- |IC| < 0.02 → ruido
- 0.02 ≤ |IC| < 0.05 → débil
- 0.05 ≤ |IC| < 0.10 → moderado
- |IC| ≥ 0.10 → fuerte

Dado que el objetivo de este stage es identificar señales con capacidad predictiva real, se descartan aquellas cuyo |IC| se encuentra por debajo del umbral de relevancia. En consecuencia, se conservan únicamente los indicadores que presentan una señal fuerte, definida como:

$$∣𝐼𝐶_Δ∣≥0.10|$$

Este primer filtro elimina indicadores dominados por ruido y reduce el espacio de features a un conjunto con relación estadísticamente significativa respecto a la magnitud del movimiento futuro.

Adicionalmente, se aplica un filtrado por familias de indicadores sobre el conjunto resultante, con el objetivo de eliminar redundancia estructural y conservar únicamente el núcleo informativo del modelo.

El orden de aplicación de los filtros es relevante: el criterio de fuerza mínima de señal se aplica antes de la reducción por familias. De este modo, la selección por familia se realiza exclusivamente sobre indicadores con capacidad predictiva comprobada, evitando retener variables redundantes cuyo IC se encuentra en el rango de ruido.

In [261]:
# Filtro por fuerza mínima de señal (|IC_delta| >= 0.10)
ic_60_relevant = ic_60[ic_60["abs_IC_delta"] >= 0.10].copy()
ic_90_relevant = ic_90[ic_90["abs_IC_delta"] >= 0.10].copy()

In [262]:
# Filtro por familia
ic_60_top_by_family = select_top_by_family(ic_60_relevant,top_n=1)
ic_90_top_by_family = select_top_by_family(ic_90_relevant,top_n=1)

In [264]:
ic_60_top_by_family

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta,family
0,price_ema60,60,-0.466803,-0.282841,-0.636263,0.015617,0.067863,,0.466803,trend_price
1,bb_60_15,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702,volatility_extension
2,rsi_14,60,-0.422538,-0.253907,-0.612055,0.012277,0.065659,,0.422538,momentum_oscillator
3,roc_60,60,-0.392433,-0.256868,-0.581905,0.028288,0.095615,,0.392433,momentum
4,macd,60,-0.333890,-0.217525,-0.535455,0.008694,0.069815,,0.333890,trend_momentum


In [265]:
ic_90_top_by_family

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta,family
0,price_ema60,90,-0.304082,-0.247576,-0.583596,-0.005852,0.033802,,0.304082,trend_price
1,rsi_14,90,-0.277105,-0.225905,-0.572693,-0.004575,0.022853,,0.277105,momentum_oscillator
2,bb_60_15,90,-0.276758,-0.227443,-0.569022,-0.002400,0.033093,,0.276758,volatility_extension
3,roc_30,90,-0.255689,-0.217309,-0.513596,-0.000619,0.016405,,0.255689,momentum
4,macd,90,-0.226918,-0.190644,-0.465284,-0.020539,0.016422,,0.226918,trend_momentum


##### **Resultados finales por horizonte**



**60 minutos**

- Set final: EMA60, BB_60_15, RSI_14, ROC_60, MACD
- `abs_IC_delta` alto (0.33–0.47): señal muy fuerte para intradía.
- `IC_trade_only` muy negativo (hasta -0.64): dirección altamente coherente cuando hay trade.
- Régimen: reversión clara y dominante.

**90 minutos**

- Set final: EMA60, BB_60_15, RSI_14, ROC_30, MACD
- `abs_IC_delta` menor (0.23–0.30): señal atenuada pero robusta.
- `IC_trade_only` sigue siendo elevado y del mismo signo: coherencia direccional preservada.
- Régimen: reversión sostenida al extender el horizonte.

**Comparación directa 60 vs 90**

- Mismas familias, mismos roles económicos.

  `trend_price`, `volatility_extension`, `momentum_oscillator`, `momentum`, `trend_momentum` aparecen en ambos horizontes.

- Diferencia solo en la escala temporal del momentum.

  - 60 min: ROC_60
  - 90 min: ROC_30

  ⇒ ajuste natural del lookback al extender el horizonte.

- Mayor potencia a 60, mayor estabilidad temporal a 90.

  60 min maximiza intensidad; 90 min mantiene estructura con más ruido.

**Conclusión operativa**

- Un único set de features por familia es válido para ambos horizontes.
- El régimen anticipatorio es de reversión en 60 y 90 minutos.
- La diferencia entre horizontes es cuantitativa, no cualitativa.

#### **3.3.3. Segundo filtro: coherencia direccional entre magnitud y dirección del movimiento**

El primer filtro garantiza que los indicadores seleccionados presentan una relación estadísticamente significativa con la magnitud del movimiento futuro. Sin embargo, una señal fuerte en magnitud no es suficiente desde el punto de vista operativo si no existe coherencia direccional con la ocurrencia efectiva del trade.

Con este objetivo, se introduce un segundo filtro basado en la comparación entre:

- $IC_{Δ}$: correlación entre el indicador técnico y la magnitud del movimiento futuro (delta en puntos).
- $IC_{trade_only}$: correlación entre el indicador técnico y la dirección del movimiento, calculada únicamente en los instantes donde efectivamente se produce un trade.

El criterio de coherencia direccional se define de la siguiente manera:

- Mismo signo entre $IC_{Δ}$ y $IC_{trade_only}$:

  La señal es direccionalmente consistente, es decir, el indicador anticipa no solo la magnitud sino también la dirección del movimiento cuando este se materializa.

- Signos opuestos entre $IC_{Δ}$ y $IC_{trade_only}$:

  La señal sugiere una mezcla de regímenes (por ejemplo, magnitud anticipada pero dirección inestable), reduciendo su utilidad operativa.

Este segundo filtro permite descartar indicadores cuya relación con la magnitud del movimiento no se traduce de forma consistente en decisiones direccionales accionables.


In [270]:
import numpy as np
import pandas as pd

def check_directional_coherence(df_selected: pd.DataFrame) -> pd.DataFrame:
    """
    Verifica coherencia direccional entre IC_delta e IC_trade_only.

    Criterio:
      - coherent = True si sign(IC_delta) == sign(IC_trade_only) y ninguno es 0/NaN
      - coherent = False en caso contrario

    Devuelve una tabla con:
      - signos, bandera de coherencia y gap de magnitudes
    """
    out = df_selected.copy()

    # Signos (NaN -> NaN)
    out["sign_IC_delta"] = np.sign(out["IC_delta"])
    out["sign_IC_trade_only"] = np.sign(out["IC_trade_only"])

    # Coherencia: mismos signos y ambos distintos de 0 y no NaN
    out["coherent"] = (
        out["IC_delta"].notna()
        & out["IC_trade_only"].notna()
        & (out["sign_IC_delta"] != 0)
        & (out["sign_IC_trade_only"] != 0)
        & (out["sign_IC_delta"] == out["sign_IC_trade_only"])
    )

    # Comparación de magnitudes (útil para ver si trade_only es más fuerte)
    out["abs_IC_trade_only"] = out["IC_trade_only"].abs()
    out["abs_IC_delta"] = out["IC_delta"].abs()  # por si no estuviera
    out["abs_gap_trade_vs_delta"] = out["abs_IC_trade_only"] - out["abs_IC_delta"]

    # Orden: primero incoherentes (si existieran), luego por señal más fuerte
    out = out.sort_values(["coherent", "abs_IC_delta"], ascending=[True, False])

    return out[
        ["family", "indicator", "IC_delta", "IC_trade_only",
         "sign_IC_delta", "sign_IC_trade_only", "coherent",
         "abs_IC_delta", "abs_IC_trade_only", "abs_gap_trade_vs_delta"]
    ]


In [274]:
coh_60 = check_directional_coherence(ic_60_top_by_family)
coh_90 = check_directional_coherence(ic_90_top_by_family)

##### **Resultados empirícos**


In [276]:
coh_60

,family,indicator,IC_delta,IC_trade_only,sign_IC_delta,sign_IC_trade_only,coherent,abs_IC_delta,abs_IC_trade_only,abs_gap_trade_vs_delta
0,trend_price,price_ema60,-0.466803,-0.636263,-1.0,-1.0,True,0.466803,0.636263,0.169460
1,volatility_extension,bb_60_15,-0.425702,-0.569178,-1.0,-1.0,True,0.425702,0.569178,0.143476
2,momentum_oscillator,rsi_14,-0.422538,-0.612055,-1.0,-1.0,True,0.422538,0.612055,0.189517
3,momentum,roc_60,-0.392433,-0.581905,-1.0,-1.0,True,0.392433,0.581905,0.189472
4,trend_momentum,macd,-0.333890,-0.535455,-1.0,-1.0,True,0.333890,0.535455,0.201565


In [277]:
coh_90

,family,indicator,IC_delta,IC_trade_only,sign_IC_delta,sign_IC_trade_only,coherent,abs_IC_delta,abs_IC_trade_only,abs_gap_trade_vs_delta
0,trend_price,price_ema60,-0.304082,-0.583596,-1.0,-1.0,True,0.304082,0.583596,0.279515
1,momentum_oscillator,rsi_14,-0.277105,-0.572693,-1.0,-1.0,True,0.277105,0.572693,0.295589
2,volatility_extension,bb_60_15,-0.276758,-0.569022,-1.0,-1.0,True,0.276758,0.569022,0.292263
3,momentum,roc_30,-0.255689,-0.513596,-1.0,-1.0,True,0.255689,0.513596,0.257907
4,trend_momentum,macd,-0.226918,-0.465284,-1.0,-1.0,True,0.226918,0.465284,0.238365


El análisis muestra que, tanto para el horizonte de 60 minutos como para el de 90 minutos:

- El 100 % de los indicadores seleccionados presentan coherencia direccional.
- En todos los casos, $IC_{Δ} < 0 $ y  $IC_{trade_only} < 0 $, confirmando un régimen de reversión consistente.
- La magnitud de $|IC_{trade_only}|$ es sistemáticamente mayor que la de $|IC_{Δ}|$, lo que indica que:
  - el indicador no solo anticipa la magnitud del movimiento,
  - sino que alinea con mayor fuerza la dirección cuando el trade ocurre efectivamente.

Este comportamiento se observa de manera uniforme en todas las familias económicas relevantes (tendencia, momentum, osciladores y volatilidad de extensión), y se mantiene al extender el horizonte temporal de 60 a 90 minutos.

**Implicancia metodológica**

La ausencia total de inconsistencias direccionales valida la utilidad operativa de los indicadores seleccionados. En particular, confirma que las señales identificadas durante la ventana de gestación no solo son estadísticamente significativas, sino también direccionalmente accionables cuando el movimiento se materializa en la ventana de expansión.


## **4. Consolidación de indicadores técnicos**

A partir de los filtros aplicados en las secciones anteriores —fuerza mínima de señal, reducción por familias y coherencia direccional— se obtiene un conjunto reducido, robusto y económicamente interpretable de indicadores técnicos.

La consolidación se apoya en tres observaciones empíricas clave:

1. Estabilidad entre horizontes

    Los horizontes de 60 y 90 minutos presentan:
    - las mismas familias económicas dominantes,
    - el mismo régimen anticipatorio (reversión),
    - y un conjunto de indicadores altamente consistente.

2. Redundancia estructural resuelta

    El filtrado por familias elimina duplicaciones (por ejemplo, múltiples EMA, variantes de Bollinger Bands y ROC), sin pérdida de señal predictiva.

3. Coherencia operativa validada

    Todos los indicadores consolidados presentan coherencia direccional entre magnitud y dirección del movimiento, con señales particularmente fuertes cuando el trade se materializa.

### **4.1. Set consolidado de indicadores**


Dado que las diferencias entre 60 y 90 minutos son cuantitativas y no cualitativas, se adopta un único set consolidado válido para ambos horizontes:

| Familia                  | Indicador              | Rol económico principal                  |
|--------------------------|------------------------|------------------------------------------|
| Tendencia de precio      | `price_ema60`          | Nivel y extensión del precio             |
| Volatilidad de extensión | `bb_60_15`             | Distancia relativa a bandas              |
| Oscilador de momentum    | `rsi_14`               | Sobrecompra / sobreventa                 |
| Momentum direccional     | `roc_60` / `roc_30`    | Velocidad del movimiento                 |
| Tendencia–momentum       | `macd`                 | Cambio de régimen                        |


Nota: para el componente de momentum, el lookback puede ajustarse al horizonte (ROC_60 para 60 min y ROC_30 para 90 min) sin alterar la estructura del modelo.

### **4.2. Implicancia para el pipeline**


Este set consolidado constituye el núcleo de features técnicas del modelo y será utilizado en las etapas posteriores para:

- entrenamiento y validación del modelo predictivo,
- análisis de contribución por feature,
- y evaluación de desempeño económico.

La consolidación reduce la dimensionalidad, mejora la interpretabilidad y alinea el diseño del modelo con la estructura temporal y económica observada en los datos.

### **4.3. Código para calcular indicadores técnicos consolidados**


In [278]:
mnq_intraday_labeled = load_data()

In [280]:
from typing import List, Tuple
import pandas as pd
import ta
from ta.volatility import BollingerBands
from ta.momentum import ROCIndicator


def compute_selected_indicators_per_day(
    df: pd.DataFrame,
    target_col: str = "close",
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Computes selected technical indicators WITHOUT crossing days.
    All indicators are calculated independently per trading day
    using groupby('date').

    Indicators computed:
      - price_ema60
      - bb_60_15
      - rsi_14
      - roc_30
      - roc_60
      - macd

    Returns
    -------
    df_out : pd.DataFrame
        DataFrame with technical indicators added.
    indicator_columns : list[str]
        List of generated indicator column names.
    """

    indicator_columns: List[str] = [
        "price_ema60",
        "bb_60_15",
        "rsi_14",
        "roc_30",
        "roc_60",
        "macd",
    ]

    def apply_per_day(day_df: pd.DataFrame) -> pd.DataFrame:
        day_df = day_df.copy()

        # EMA-based price extension (normalized)
        day_df["price_ema60"] = (
            day_df[target_col] / day_df[target_col].ewm(span=60).mean() - 1
        )

        # Bollinger %B (window=60, deviation=1.5)
        day_df["bb_60_15"] = BollingerBands(
            day_df[target_col],
            window=60,
            window_dev=1.5
        ).bollinger_pband()

        # RSI (14)
        day_df["rsi_14"] = ta.momentum.RSIIndicator(
            day_df[target_col],
            window=14
        ).rsi()

        # Rate of Change (30, 60)
        day_df["roc_30"] = ROCIndicator(
            close=day_df[target_col],
            window=30
        ).roc()

        day_df["roc_60"] = ROCIndicator(
            close=day_df[target_col],
            window=60
        ).roc()

        # MACD difference
        day_df["macd"] = ta.trend.MACD(
            day_df[target_col]
        ).macd_diff()

        return day_df

    df_out = df.groupby("date", group_keys=False).apply(apply_per_day)

    return df_out, indicator_columns


In [286]:
mnq_features_target, mnq_technical_indicators_list = compute_selected_indicators_per_day(mnq_intraday_labeled)

In [287]:
mnq_features_target.head()

,date,open,high,low,close,volume,delta_pts_60,trade_60,target_op_60,target_tail_60,delta_pts_90,trade_90,target_op_90,target_tail_90,price_ema60,bb_60_15,rsi_14,roc_30,roc_60,macd
datetime,,,,,,,,,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,9.00,0,0,0,6.00,0,0,0,0.000000,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,9.25,0,0,0,7.50,0,0,0,-0.000070,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,9.75,0,0,0,8.00,0,0,0,-0.000140,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,8.50,0,0,0,8.00,0,0,0,-0.000040,NaN,NaN,NaN,NaN,NaN
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,8.00,0,0,0,7.75,0,0,0,-0.000031,NaN,NaN,NaN,NaN,NaN


In [288]:
info_dataset(mnq_features_target)

# Eliminar filas con al menos un NaN
mnq_features_target = mnq_features_target.dropna()

info_dataset(mnq_features_target)


Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York
Información del dataset:

	Cantidad de días: 1303
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York


## **4. Guardado de dataset de indicadores técnicos**

### 4.1. Preparamos el dataset mnq_technical_indicators



In [230]:
columnas_base = ['date', 'open', 'high', 'low','close','volume']
columnas_target = ['target_return_30','target_return_60','target_return_90'] #Mantenemos solo los targets de interés
columnas_indicadores = ['rsi_14', 'rsi_7', 'momentum_10', 'momentum_5', 'macd', 'price_ema20', 'price_ema30', 'stoch_k_20', 'stoch_k_30',  'bb_20', 'bb_30', 'bb_60', 'atr_norm','roc_20', 'roc_30', 'roc_60']

In [231]:
#Definir la lista de datasets
mnq_technical_indicators = [mnq_rsi, mnq_momentum, mnq_volumen, mnq_macd, mnq_ema,
              mnq_stoch, mnq_bollinger, mnq_atr, mnq_roc]

#Concatenar horizontalmente
mnq_technical_indicators = pd.concat(mnq_technical_indicators, axis=1)

#Eliminar columnas duplicadas (date, OHLCV, targets)
mnq_technical_indicators = mnq_technical_indicators.loc[:, ~mnq_technical_indicators.columns.duplicated()]

# Seleccionar columnas
mnq_technical_indicators = mnq_technical_indicators[ columnas_base + columnas_target + columnas_indicadores].copy()

NameError: name 'mnq_momentum' is not defined

In [ ]:
# Guardar como Parquet
mnq_technical_indicators.to_parquet(f'{drive_path}/2_feature_engineering/mnq_technical_indicators.parquet')

In [ ]:
path = f"{drive_path}/2_feature_engineering/mnq_technical_indicators.parquet"

if os.path.exists(path):
    print("✔️ El archivo existe en disco:", path)
else:
    print("❌ El archivo NO existe.")

df_check = pd.read_parquet(path)
print("✔️ Archivo leído correctamente. Filas/columnas:", df_check.shape)

# Comparación básica de forma
print("Original:", mnq_technical_indicators.shape)
print("Leído   :", df_check.shape)

is_equal = mnq_technical_indicators.equals(df_check)
print("¿Los DataFrames son idénticos?:", is_equal)

### 4.2. Preparamos el dataset IC_technical_indicators


In [ ]:
# Columnas a eliminar
cols_drop = ["ic_mean_5min", "ic_std_5min", "ic_mean_15min", "ic_std_15min"]
# Eliminamos las columnas
ic_technical_indicators = IC_indicadores.drop(columns=cols_drop, errors="ignore")

In [ ]:
technical_indicators_features = ['rsi_14', 'rsi_7', 'momentum_10', 'momentum_5', 'macd', 'price_ema20', 'price_ema30', 'stoch_k_20', 'stoch_k_30',  'bb_20', 'bb_30', 'bb_60', 'atr_norm','roc_20', 'roc_30', 'roc_60']
ic_technical_indicators = ic_technical_indicators[ic_technical_indicators["indicador"].isin(technical_indicators_features)].copy()

In [ ]:
# Guardar como Parquet
ic_technical_indicators.to_parquet(f'{drive_path}/2_feature_engineering/ic_technical_indicators.parquet')